Block 1 — Load Part B + NB0 artifacts, imports (with robust path auto-detection)

In [ ]:
import sys, os

NB0_DIR   = "/kaggle/input/notebooks/sumiayasharif/nb0-eda-data-prep/nb0_artifacts"
PARTB_DIR = "/kaggle/input/notebooks/sumiayasharif/data-prep-1/partb_dataprep_artifacts"

for d, name in [(NB0_DIR, "NB0_DIR"), (PARTB_DIR, "PARTB_DIR")]:
    if not os.path.exists(d):
        print(f"{name} NOT FOUND at {d} — walking /kaggle/input:")
        for root, dirs, files in os.walk("/kaggle/input"):
            if "seg_common.py" in files or "ssl_common.py" in files:
                print(" ", root, files)
    else:
        print(f"{name} OK:", d, "->", os.listdir(d))

sys.path.insert(0, NB0_DIR)
sys.path.insert(0, PARTB_DIR)

import gc, time, json, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torchvision.models as tv_models
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2

from ssl_common import load_partb_config, load_remapped_split
from seg_common import FoodSegDataset, get_transforms

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("Total memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

partb_cfg = load_partb_config(f"{PARTB_DIR}/partb_config.json")
split = load_remapped_split(f"{PARTB_DIR}/split_manifest_remapped.json")

NUM_CLASSES = partb_cfg["num_classes_total"]
CLASS_NAMES = ["background"] + partb_cfg["class_names"]

print(f"Train (unlabelled, Task B corpus): {len(split['train'])}")
print(f"Valid (labelled fine-tune, Task D): {len(split['valid'])}")
print(f"Test (monitor + final eval, Tasks D&E): {len(split['test'])}")

Block 1b — Auto-detect and remap raw dataset paths (in case the dataset owner/slug changed for this session)

In [ ]:
sample_paths = split["train"][:3]
needs_remap = not all(os.path.exists(p) for p in sample_paths)
print("Needs remap:", needs_remap)

if needs_remap:
    print("Recorded paths don't resolve. Searching /kaggle/input for the raw FoodBD dataset...")
    detected_root = None
    for root, dirs, files in os.walk("/kaggle/input"):
        if root.endswith(os.path.join("train", "images")) and any(f.endswith(".jpg") for f in files):
            # root looks like .../FoodBD/train/images -> the FoodBD root is two levels up
            detected_root = os.path.dirname(os.path.dirname(root))
            print("Detected FoodBD root:", detected_root)
            break

    assert detected_root is not None, (
        "Could not auto-detect the raw FoodBD dataset under /kaggle/input. "
        "Attach it via Add Input -> Datasets, then re-run this cell."
    )

    def remap_path(old_path, real_root):
        for split_name in ("train", "valid", "test"):
            marker = f"/{split_name}/images/"
            if marker in old_path:
                tail = old_path.split(marker, 1)[1]
                return os.path.join(real_root, split_name, "images", tail)
        return old_path

    for split_name in ["train", "valid", "test"]:
        split[split_name] = [remap_path(p, detected_root) for p in split[split_name]]

    for p in split["train"][:3]:
        print(p, "-> exists:", os.path.exists(p))

print(f"\nFinal split sizes -> train={len(split['train'])}, valid={len(split['valid'])}, "
      f"test={len(split['test'])}")

Block 2 — Config cell

In [ ]:
CONFIG = {
    "method": "SimCLR",
    "backbone": "ResNet-50",
    "decoder": "SegFormer-style All-MLP head (multi-scale ResNet features -> project -> concat -> fuse -> classify)",
    "decoder_source": "Re-implemented from SegFormer's All-MLP decode head, per Part A's best-performing design (SegFormer-B0)",
    "checkpoint_path": "torchvision ImageNet1K_V2 (continue SSL pretraining from this checkpoint)",
    "compute_budget_path": "Path 1 — continue SSL pretraining from an ImageNet-pretrained checkpoint",
    "pretrain_img_size": 224,
    "pretrain_batch_size": 32,
    "gradient_accumulation_steps": 2,
    "pretrain_epochs_required": 50,
    "temperature": 0.5,
    "projection_dim": 128,
    "optimizer": "AdamW",
    "lr": 3e-4,
    "weight_decay": 1e-6,
    "lr_schedule": "CosineAnnealingLR",
    "mixed_precision": True,
    "seed": SEED,
}
print(json.dumps(CONFIG, indent=2))

Block 3 — SimCLR two-view augmentation + unlabelled dataset

In [ ]:
simclr_transform = A.Compose([
    A.RandomResizedCrop(size=(CONFIG["pretrain_img_size"], CONFIG["pretrain_img_size"]),
                         scale=(0.2, 1.0), p=1.0),
    A.HorizontalFlip(p=0.5),
    A.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1, p=0.8),
    A.ToGray(p=0.2),
    A.GaussianBlur(blur_limit=(3, 7), p=0.5),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2(),
])

class SimCLRPretrainDataset(Dataset):
    def __init__(self, img_paths, transform):
        self.img_paths = img_paths
        self.transform = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img = cv2.cvtColor(cv2.imread(str(self.img_paths[idx])), cv2.COLOR_BGR2RGB)
        view1 = self.transform(image=img)["image"]
        view2 = self.transform(image=img)["image"]
        return view1, view2

pretrain_ds = SimCLRPretrainDataset(split["train"], simclr_transform)

pretrain_loader = DataLoader(
    pretrain_ds,
    batch_size=CONFIG["pretrain_batch_size"],
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True,
    persistent_workers=False,
)
print(f"Pretraining batches per epoch: {len(pretrain_loader)}")

v1, v2 = next(iter(pretrain_loader))
print("View shapes:", v1.shape, v2.shape)
del v1, v2
gc.collect(); torch.cuda.empty_cache()

Block 4 — SimCLR model (ResNet-50 encoder + projection head)

In [ ]:
class SimCLREncoder(nn.Module):
    def __init__(self, projection_dim=128):
        super().__init__()
        backbone = tv_models.resnet50(weights=tv_models.ResNet50_Weights.IMAGENET1K_V2)
        self.feature_dim = backbone.fc.in_features  # 2048
        backbone.fc = nn.Identity()
        self.encoder = backbone

        self.projector = nn.Sequential(
            nn.Linear(self.feature_dim, self.feature_dim),
            nn.ReLU(inplace=True),
            nn.Linear(self.feature_dim, projection_dim),
        )

    def forward(self, x):
        h = self.encoder(x)
        z = self.projector(h)
        return h, z

model = SimCLREncoder(projection_dim=CONFIG["projection_dim"]).to(device)
print(f"Total params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

Block 5 — Memory-safe, dtype-safe NT-Xent loss

In [ ]:
def nt_xent_loss(z1, z2, temperature=0.5):
    N = z1.size(0)
    z = torch.cat([z1, z2], dim=0)
    z = F.normalize(z, dim=1)
    sim = torch.mm(z, z.t()) / temperature

    mask = torch.eye(2*N, dtype=torch.bool, device=z.device)
    fill_value = torch.finfo(sim.dtype).min   # dtype-safe (avoids float16 overflow under autocast)
    sim = sim.masked_fill(mask, fill_value)

    pos_idx = torch.cat([torch.arange(N, 2*N), torch.arange(0, N)]).to(z.device)
    positives = sim[torch.arange(2*N), pos_idx]

    loss = -positives + torch.logsumexp(sim, dim=1)
    return loss.mean()

Block 6 — Task B: SimCLR pretraining loop (50 epochs, current torch.amp API)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG["pretrain_epochs_required"])
scaler = torch.amp.GradScaler('cuda', enabled=CONFIG["mixed_precision"])

ACCUM_STEPS = CONFIG["gradient_accumulation_steps"]
pretrain_history = {"loss": [], "epoch_time_sec": []}
EPOCH_TIME_LIMIT_SEC = 600
split_warning_shown = False

for epoch in range(1, CONFIG["pretrain_epochs_required"] + 1):
    model.train()
    epoch_start = time.time()
    running_loss = 0.0
    optimizer.zero_grad()

    for step, (v1, v2) in enumerate(pretrain_loader):
        v1, v2 = v1.to(device, non_blocking=True), v2.to(device, non_blocking=True)

        with torch.amp.autocast('cuda', enabled=CONFIG["mixed_precision"]):
            _, z1 = model(v1)
            _, z2 = model(v2)
            loss = nt_xent_loss(z1, z2, temperature=CONFIG["temperature"]) / ACCUM_STEPS

        scaler.scale(loss).backward()

        if (step + 1) % ACCUM_STEPS == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        running_loss += loss.item() * ACCUM_STEPS

        del v1, v2, z1, z2, loss
        if step % 50 == 0:
            torch.cuda.empty_cache()

    scheduler.step()
    epoch_time = time.time() - epoch_start
    avg_loss = running_loss / len(pretrain_loader)
    pretrain_history["loss"].append(avg_loss)
    pretrain_history["epoch_time_sec"].append(epoch_time)

    print(f"[SimCLR Pretrain] Epoch {epoch:03d}/{CONFIG['pretrain_epochs_required']} | "
          f"loss={avg_loss:.4f} | time={epoch_time:.1f}s")

    if epoch == 1 and epoch_time > EPOCH_TIME_LIMIT_SEC and not split_warning_shown:
        print(f"\n*** WARNING: epoch time {epoch_time:.0f}s exceeds 600s (10 min). "
              "Consider splitting into pretrain + downstream notebooks. ***\n")
        split_warning_shown = True

    torch.save(model.encoder.state_dict(), "simclr_encoder_latest.pth")
    gc.collect()
    torch.cuda.empty_cache()

CONFIG["actual_pretrain_epochs"] = epoch
CONFIG["avg_epoch_time_sec"] = float(np.mean(pretrain_history["epoch_time_sec"]))
CONFIG["total_pretrain_time_sec"] = float(np.sum(pretrain_history["epoch_time_sec"]))
print(f"\nPretraining complete. Avg epoch time: {CONFIG['avg_epoch_time_sec']:.1f}s, "
      f"total: {CONFIG['total_pretrain_time_sec']/60:.1f} min")

torch.save(model.encoder.state_dict(), "simclr_encoder_final.pth")

Block 7 — Pretext loss curve

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(pretrain_history["loss"])
plt.xlabel("Epoch"); plt.ylabel("NT-Xent Loss")
plt.title(f"SimCLR Pretraining Loss ({CONFIG['actual_pretrain_epochs']} epochs)")
plt.tight_layout()
plt.savefig("simclr_pretrain_loss_curve.png", dpi=150)
plt.show()

Block 8 — Free pretraining-stage memory

In [ ]:
del pretrain_loader, pretrain_ds
gc.collect()
torch.cuda.empty_cache()
print("Pretraining objects freed. GPU memory reserved (GB):", round(torch.cuda.memory_reserved()/1e9, 2))

Block 9 — Task C: SegFormer-style All-MLP decoder

In [ ]:
class SegformerStyleMLPHead(nn.Module):
    """Re-implementation of SegFormer's All-MLP decode head, adapted to ResNet-50's
    multi-scale feature maps (in_channels_list = [256, 512, 1024, 2048] at strides [4,8,16,32]),
    matching the design identified as best-performing in Part A (SegFormer-B0)."""
    def __init__(self, in_channels_list, decoder_hidden_size=256, num_classes=NUM_CLASSES, dropout=0.1):
        super().__init__()
        # One 1x1 "linear" projection per scale (implemented as conv for spatial inputs)
        self.linear_projs = nn.ModuleList([
            nn.Conv2d(c, decoder_hidden_size, kernel_size=1) for c in in_channels_list
        ])
        self.fuse = nn.Sequential(
            nn.Conv2d(decoder_hidden_size * len(in_channels_list), decoder_hidden_size, kernel_size=1),
            nn.BatchNorm2d(decoder_hidden_size),
            nn.ReLU(inplace=True),
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Conv2d(decoder_hidden_size, num_classes, kernel_size=1)

    def forward(self, feats, out_size):
        # feats: list of 4 feature maps [layer1, layer2, layer3, layer4], highest-res first
        target_hw = feats[0].shape[-2:]  # upsample everything to layer1's resolution (stride 4)
        projected = []
        for feat, proj in zip(feats, self.linear_projs):
            p = proj(feat)
            p = F.interpolate(p, size=target_hw, mode="bilinear", align_corners=False)
            projected.append(p)
        x = torch.cat(projected, dim=1)
        x = self.fuse(x)
        x = self.dropout(x)
        x = self.classifier(x)
        return F.interpolate(x, size=out_size, mode="bilinear", align_corners=False)


class SimCLRSegFormerStyleModel(nn.Module):
    """Encoder (SimCLR-pretrained ResNet-50, multi-scale feature maps) + SegFormer-style All-MLP decoder."""
    def __init__(self, encoder, num_classes=NUM_CLASSES, freeze_encoder=False):
        super().__init__()
        self.stem0 = nn.Sequential(encoder.conv1, encoder.bn1, encoder.relu, encoder.maxpool)
        self.layer1 = encoder.layer1  # stride 4,  256 channels
        self.layer2 = encoder.layer2  # stride 8,  512 channels
        self.layer3 = encoder.layer3  # stride 16, 1024 channels
        self.layer4 = encoder.layer4  # stride 32, 2048 channels

        self.decoder = SegformerStyleMLPHead(
            in_channels_list=[256, 512, 1024, 2048],
            decoder_hidden_size=256,
            num_classes=num_classes,
        )
        self.freeze_encoder = freeze_encoder
        if freeze_encoder:
            for m in [self.stem0, self.layer1, self.layer2, self.layer3, self.layer4]:
                for p in m.parameters():
                    p.requires_grad = False

    def forward(self, x):
        x0 = self.stem0(x)
        f1 = self.layer1(x0)   # stride 4
        f2 = self.layer2(f1)   # stride 8
        f3 = self.layer3(f2)   # stride 16
        f4 = self.layer4(f3)   # stride 32
        return self.decoder([f1, f2, f3, f4], out_size=x.shape[-2:])


pretrained_encoder = tv_models.resnet50(weights=None)
pretrained_encoder.fc = nn.Identity()   # matches architecture used when the checkpoint was saved
pretrained_encoder.load_state_dict(torch.load("simclr_encoder_final.pth"))
print("Encoder checkpoint loaded successfully.")

Block 10 — Task C strategy: reuse the frozen-vs-fine-tuned decision from the earlier ASPP comparison

In [ ]:
# Frozen-vs-fine-tuned was already compared for SimCLR+ASPP earlier (frozen won: 0.1578 vs 0.1485).
# Swapping only the decoder design doesn't change which strategy suits a small labelled set better,
# so we reuse that decision here rather than re-running the probe.
FREEZE_ENCODER = True
print(f"Reusing Task C decision: encoder_frozen={FREEZE_ENCODER}")

seg_model = SimCLRSegFormerStyleModel(pretrained_encoder, num_classes=NUM_CLASSES,
                                       freeze_encoder=FREEZE_ENCODER).to(device)

print(f"Segmentation model params: {sum(p.numel() for p in seg_model.parameters())/1e6:.1f}M")
print(f"Trainable params: {sum(p.numel() for p in seg_model.parameters() if p.requires_grad)/1e6:.1f}M")

Block 11 — Task D: Fine-tuning datasets + config

In [ ]:
FT_IMG_SIZE = 384
finetune_transform = get_transforms(FT_IMG_SIZE, train=True)
monitor_transform  = get_transforms(FT_IMG_SIZE, train=False)

finetune_ds = FoodSegDataset(split["valid"], finetune_transform, bg_is_zero=partb_cfg["bg_is_zero"])
monitor_ds  = FoodSegDataset(split["test"],  monitor_transform,  bg_is_zero=partb_cfg["bg_is_zero"])

FT_BATCH_SIZE = 8
finetune_loader = DataLoader(finetune_ds, batch_size=FT_BATCH_SIZE, shuffle=True,
                              num_workers=2, pin_memory=True, drop_last=True)
monitor_loader  = DataLoader(monitor_ds,  batch_size=FT_BATCH_SIZE, shuffle=False,
                              num_workers=2, pin_memory=True)

print(f"Fine-tune set (validation-split, labelled): {len(finetune_ds)} images")
print(f"Monitor/eval set (test-split, double role): {len(monitor_ds)} images")

dist_df = pd.read_csv(f"{NB0_DIR}/pixel_class_distribution.csv").sort_values("class_id").reset_index(drop=True)
pixel_counts = np.clip(dist_df["pixel_count"].values.astype(np.float64), 1, None)
inv_sqrt_freq = 1.0 / np.sqrt(pixel_counts)
class_weights = inv_sqrt_freq / inv_sqrt_freq.sum() * NUM_CLASSES
class_weights = np.clip(class_weights, 0.1, 10.0)
class_weights[0] *= 0.5
class_weights_t = torch.tensor(class_weights, dtype=torch.float32).to(device)

def dice_loss(logits, targets, num_classes, eps=1e-6):
    probs = F.softmax(logits, dim=1)
    targets_onehot = F.one_hot(targets, num_classes).permute(0, 3, 1, 2).float()
    dims = (0, 2, 3)
    intersection = torch.sum(probs * targets_onehot, dims)
    union = torch.sum(probs + targets_onehot, dims)
    dice = (2 * intersection + eps) / (union + eps)
    return 1 - dice.mean()

ce_loss_fn = nn.CrossEntropyLoss(weight=class_weights_t)

def combo_loss(logits, targets):
    return 0.6 * ce_loss_fn(logits, targets) + 0.4 * dice_loss(logits, targets, NUM_CLASSES)

def compute_miou(conf_mat, num_classes):
    ious = []
    for c in range(num_classes):
        tp = conf_mat[c, c]
        fp = conf_mat[:, c].sum() - tp
        fn = conf_mat[c, :].sum() - tp
        denom = tp + fp + fn
        ious.append(tp / denom if denom > 0 else np.nan)
    return np.nanmean(ious), ious

def update_confusion(conf_mat, preds, targets, num_classes):
    p = preds.flatten().cpu().numpy()
    t = targets.flatten().cpu().numpy()
    mask = (t >= 0) & (t < num_classes)
    conf_mat += np.bincount(
        num_classes * t[mask].astype(int) + p[mask].astype(int),
        minlength=num_classes**2
    ).reshape(num_classes, num_classes)
    return conf_mat

FT_CONFIG = {
    "stage": "Task D - Supervised Fine-tuning",
    "finetune_img_size": FT_IMG_SIZE,
    "finetune_batch_size": FT_BATCH_SIZE,
    "finetune_epochs_required": 50,
    "encoder_lr": 1e-5,
    "decoder_lr": 1e-4,
    "optimizer": "AdamW",
    "weight_decay": 1e-4,
    "loss_function": "Weighted CrossEntropy + Dice (combo)",
    "encoder_frozen": bool(FREEZE_ENCODER),
    "labelled_images_used": int(len(finetune_ds)),
}
print(json.dumps(FT_CONFIG, indent=2))

Block 12 — Task D: Full fine-tuning loop (50 epochs)

In [ ]:
if FREEZE_ENCODER:
    param_groups = [{"params": seg_model.decoder.parameters(), "lr": FT_CONFIG["decoder_lr"]}]
else:
    encoder_params = list(seg_model.stem0.parameters()) + list(seg_model.layer1.parameters()) + \
                      list(seg_model.layer2.parameters()) + list(seg_model.layer3.parameters()) + \
                      list(seg_model.layer4.parameters())
    param_groups = [
        {"params": encoder_params, "lr": FT_CONFIG["encoder_lr"]},
        {"params": seg_model.decoder.parameters(), "lr": FT_CONFIG["decoder_lr"]},
    ]

ft_optimizer = torch.optim.AdamW(param_groups, weight_decay=FT_CONFIG["weight_decay"])
ft_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(ft_optimizer, T_max=FT_CONFIG["finetune_epochs_required"])
ft_scaler = torch.amp.GradScaler('cuda')

ft_history = {"train_loss": [], "monitor_loss": [], "monitor_miou": [], "epoch_time_sec": []}
best_monitor_miou = -1.0
best_epoch = 0

def set_encoder_eval(m):
    m.stem0.eval(); m.layer1.eval(); m.layer2.eval(); m.layer3.eval(); m.layer4.eval()

for epoch in range(1, FT_CONFIG["finetune_epochs_required"] + 1):
    epoch_start = time.time()
    seg_model.train()
    if FREEZE_ENCODER:
        set_encoder_eval(seg_model)
    running_loss = 0.0

    for imgs, masks in finetune_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        ft_optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            logits = seg_model(imgs)
            loss = combo_loss(logits, masks)
        ft_scaler.scale(loss).backward()
        ft_scaler.step(ft_optimizer)
        ft_scaler.update()
        running_loss += loss.item()
        del imgs, masks, logits, loss

    ft_scheduler.step()
    train_loss = running_loss / len(finetune_loader)

    seg_model.eval()
    monitor_loss = 0.0
    conf_mat = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
    with torch.no_grad():
        for imgs, masks in monitor_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            logits = seg_model(imgs)
            monitor_loss += ce_loss_fn(logits, masks).item()
            preds = logits.argmax(1)
            conf_mat = update_confusion(conf_mat, preds, masks, NUM_CLASSES)
            del imgs, masks, logits, preds
    monitor_loss /= len(monitor_loader)
    monitor_miou, _ = compute_miou(conf_mat, NUM_CLASSES)

    epoch_time = time.time() - epoch_start
    ft_history["train_loss"].append(train_loss)
    ft_history["monitor_loss"].append(monitor_loss)
    ft_history["monitor_miou"].append(monitor_miou)
    ft_history["epoch_time_sec"].append(epoch_time)

    print(f"[SimCLR+SegFormerHead Fine-tune] Epoch {epoch:03d}/{FT_CONFIG['finetune_epochs_required']} | "
          f"train_loss={train_loss:.4f} | monitor_loss={monitor_loss:.4f} | "
          f"monitor_mIoU={monitor_miou:.4f} | time={epoch_time:.1f}s")

    if epoch == 1 and epoch_time > 600:
        print(f"\n*** WARNING: fine-tune epoch time {epoch_time:.0f}s exceeds 10 min. "
              "Consider splitting into a downstream notebook. ***\n")

    if monitor_miou > best_monitor_miou:
        best_monitor_miou = monitor_miou
        best_epoch = epoch
        torch.save(seg_model.state_dict(), "simclr_segformerhead_best.pth")

    gc.collect(); torch.cuda.empty_cache()

FT_CONFIG["actual_finetune_epochs"] = epoch
FT_CONFIG["best_epoch"] = best_epoch
FT_CONFIG["final_epoch_monitor_miou"] = float(ft_history["monitor_miou"][-1])
FT_CONFIG["avg_epoch_time_sec"] = float(np.mean(ft_history["epoch_time_sec"]))
FT_CONFIG["total_finetune_time_sec"] = float(np.sum(ft_history["epoch_time_sec"]))
print(f"\nFine-tuning complete after {epoch} epochs. "
      f"Best monitor mIoU={best_monitor_miou:.4f} at epoch {best_epoch}. "
      f"Final-epoch mIoU={FT_CONFIG['final_epoch_monitor_miou']:.4f}")

Block 13 — Fine-tuning curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ft_history["train_loss"], label="train_loss (validation-split, labelled)")
axes[0].plot(ft_history["monitor_loss"], label="monitor_loss (test-split)")
axes[0].set_title("Fine-tuning Loss"); axes[0].legend()
axes[1].plot(ft_history["monitor_miou"], color="green", label="monitor mIoU (test-split)")
axes[1].axvline(best_epoch-1, color="red", linestyle="--", label=f"best epoch {best_epoch}")
axes[1].set_title("Monitoring mIoU"); axes[1].legend()
plt.tight_layout()
plt.savefig("simclr_segformerhead_finetune_curves.png", dpi=150)
plt.show()

Block 14 — Task E: Final test-set evaluation

In [ ]:
seg_model.load_state_dict(torch.load("simclr_segformerhead_best.pth"))
seg_model.eval()

conf_mat_test = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
per_image_ious = []

with torch.no_grad():
    for batch_idx, (imgs, masks) in enumerate(monitor_loader):
        imgs, masks = imgs.to(device), masks.to(device)
        logits = seg_model(imgs)
        preds = logits.argmax(1)
        conf_mat_test = update_confusion(conf_mat_test, preds, masks, NUM_CLASSES)

        for i in range(imgs.size(0)):
            p, t = preds[i], masks[i]
            classes_present = torch.unique(t)
            ious = []
            for c in classes_present:
                c = c.item()
                if c == 0:
                    continue
                tp = ((p == c) & (t == c)).sum().item()
                fp_fn = ((p == c) | (t == c)).sum().item()
                if fp_fn > 0:
                    ious.append(tp / fp_fn)
            img_iou = np.mean(ious) if ious else np.nan
            per_image_ious.append((batch_idx * FT_BATCH_SIZE + i, img_iou))
        del imgs, masks, logits, preds

    gc.collect(); torch.cuda.empty_cache()

miou_test, per_class_iou = compute_miou(conf_mat_test, NUM_CLASSES)
pixel_acc = np.trace(conf_mat_test) / conf_mat_test.sum()
per_class_acc = [conf_mat_test[c,c]/conf_mat_test[c,:].sum() if conf_mat_test[c,:].sum()>0 else np.nan
                  for c in range(NUM_CLASSES)]
mean_pixel_acc = np.nanmean(per_class_acc)

def dice_from_conf(conf_mat, num_classes):
    dices = []
    for c in range(num_classes):
        tp = conf_mat[c,c]; fp = conf_mat[:,c].sum()-tp; fn = conf_mat[c,:].sum()-tp
        denom = 2*tp+fp+fn
        dices.append(2*tp/denom if denom>0 else np.nan)
    return dices

per_class_dice = dice_from_conf(conf_mat_test, NUM_CLASSES)

results_df = pd.DataFrame({
    "class_id": range(NUM_CLASSES), "class_name": CLASS_NAMES,
    "IoU": per_class_iou, "PixelAcc": per_class_acc, "Dice": per_class_dice,
})
results_df.to_csv("simclr_segformerhead_test_metrics_per_class.csv", index=False)

summary = {
    "method": "SimCLR",
    "decoder": "SegFormer-style All-MLP head",
    "mIoU": float(miou_test),
    "overall_pixel_accuracy": float(pixel_acc),
    "mean_pixel_accuracy": float(mean_pixel_acc),
    "mean_dice": float(np.nanmean(per_class_dice)),
    "pretrain_epochs": int(CONFIG["actual_pretrain_epochs"]),
    "pretrain_avg_epoch_time_sec": float(CONFIG["avg_epoch_time_sec"]),
    "pretrain_total_time_sec": float(CONFIG["total_pretrain_time_sec"]),
    "finetune_epochs": int(FT_CONFIG["actual_finetune_epochs"]),
    "finetune_avg_epoch_time_sec": float(FT_CONFIG["avg_epoch_time_sec"]),
    "finetune_total_time_sec": float(FT_CONFIG["total_finetune_time_sec"]),
    "labelled_images_used": int(FT_CONFIG["labelled_images_used"]),
    "encoder_frozen": bool(FREEZE_ENCODER),
    "test_split_double_role_note": "Test split used for both checkpoint monitoring (Task D) and "
                                    "final metrics (Task E) — see Part B Section 2 caveat.",
}
with open("simclr_segformerhead_results.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
results_df.sort_values("IoU").head(15)

Block 15 — Confusion matrix

In [ ]:
plt.figure(figsize=(16,14))
top_classes = dist_df.sort_values("pixel_count", ascending=False)["class_id"].values[:30]
sub_conf = conf_mat_test[np.ix_(top_classes, top_classes)]
sub_conf_norm = sub_conf / (sub_conf.sum(axis=1, keepdims=True) + 1e-9)

plt.imshow(sub_conf_norm, cmap="viridis")
plt.colorbar(label="Row-normalized frequency")
plt.xticks(range(len(top_classes)), [CLASS_NAMES[c] for c in top_classes], rotation=90, fontsize=7)
plt.yticks(range(len(top_classes)), [CLASS_NAMES[c] for c in top_classes], fontsize=7)
plt.title("SimCLR + SegFormer-style Head — Pixel Confusion Matrix (Top-30 classes)")
plt.tight_layout()
plt.savefig("simclr_segformerhead_confusion_matrix.png", dpi=150)
plt.show()

Block 16 — Task F: Error analysis

In [ ]:
per_image_ious_sorted = sorted(per_image_ious, key=lambda x: (np.nan if np.isnan(x[1]) else x[1]))
worst_5 = [x for x in per_image_ious_sorted if not np.isnan(x[1])][:5]
print("Worst 5 test images:")
for idx, iou in worst_5:
    print(f"  idx={idx}, IoU={iou:.4f}")

def denorm(img_t):
    mean = np.array([0.485,0.456,0.406]).reshape(3,1,1)
    std  = np.array([0.229,0.224,0.225]).reshape(3,1,1)
    return np.clip((img_t.numpy()*std + mean).transpose(1,2,0), 0, 1)

fig, axes = plt.subplots(len(worst_5), 3, figsize=(10, 4*len(worst_5)))
seg_model.eval()
with torch.no_grad():
    for row, (idx, iou) in enumerate(worst_5):
        img_t, mask_t = monitor_ds[idx]
        pred = seg_model(img_t.unsqueeze(0).to(device)).argmax(1).squeeze(0).cpu().numpy()
        img_np = denorm(img_t)
        axes[row,0].imshow(img_np); axes[row,0].set_title(f"idx={idx}"); axes[row,0].axis("off")
        axes[row,1].imshow(mask_t.numpy(), cmap="tab20", vmin=0, vmax=NUM_CLASSES); axes[row,1].axis("off")
        axes[row,2].imshow(pred, cmap="tab20", vmin=0, vmax=NUM_CLASSES); axes[row,2].set_title(f"IoU={iou:.3f}"); axes[row,2].axis("off")

plt.suptitle("SimCLR + SegFormer-style Head — Worst 5 Test Predictions vs Ground Truth")
plt.tight_layout()
plt.savefig("simclr_segformerhead_worst_predictions.png", dpi=150)
plt.show()

with open("simclr_segformerhead_worst_indices.json", "w") as f:
    json.dump([idx for idx, _ in worst_5], f)

Block 17 — Save outputs for the Final Comparison notebook

In [ ]:
import shutil
os.makedirs("/kaggle/working/simclr_segformerhead_outputs", exist_ok=True)
for f in ["simclr_segformerhead_results.json", "simclr_segformerhead_test_metrics_per_class.csv",
          "simclr_segformerhead_worst_indices.json", "simclr_segformerhead_best.pth",
          "simclr_encoder_final.pth", "simclr_pretrain_loss_curve.png",
          "simclr_segformerhead_finetune_curves.png", "simclr_segformerhead_confusion_matrix.png",
          "simclr_segformerhead_worst_predictions.png"]:
    shutil.copy(f, f"/kaggle/working/simclr_segformerhead_outputs/{f}")
print("Outputs saved:", os.listdir("/kaggle/working/simclr_segformerhead_outputs"))